# Pixel–Motif–Graph Hướng 1 — CRS Train-only stage

Source-locked execution for Issue #84 / Draft PR #85. This notebook fits only the registered Training-side primitive/composition dictionaries and P/M/C linear probes. It has no PublicTest or PrivateTest input. Scientific success is not inferred from Train metrics.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Irthn1311/FER2013_Graph.git'
REPO_BRANCH = 'research/pixel-relational-composition-crs'
SOURCE_SHA = '194876ccb5a723255f6e6ffaf3105f8cc3f1fc9e'
EXPECTED_DICTIONARY_SHA256 = '68154a054f712bb07692146904bcba57f10e079c7efc92723aa0bccba9f6273b'
FER_ROOT = Path('/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split')
TRAIN_CSV = FER_ROOT / 'train.csv'
OUTPUT_DIR = Path('/kaggle/working/outputs/pixel_relational_composition_crs_train')
PACKAGE_RELATIVE = Path('research/pixel_relational_motif_e0')
RUN_TESTS = True
RUN_TRAIN_STAGE = True


In [ ]:
import hashlib, json, os, platform, shutil, subprocess, sys
WORKING = Path('/kaggle/working')
PROJECT = WORKING / 'FER2013_Graph_CRS_TRAIN'
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(PROJECT)], check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',SOURCE_SHA], check=True)
head = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'], text=True).strip()
if head != SOURCE_SHA: raise RuntimeError(f'source lock mismatch: {head} != {SOURCE_SHA}')
subprocess.run(['git','-C',str(PROJECT),'diff','--quiet'], check=True)
subprocess.run(['git','-C',str(PROJECT),'diff','--cached','--quiet'], check=True)
PACKAGE = PROJECT / PACKAGE_RELATIVE
PACKAGE_SRC = PACKAGE / 'src'
sys.path.insert(0, str(PACKAGE_SRC))
import pixel_relational_motif_e0
imported = Path(pixel_relational_motif_e0.__file__).resolve()
if PACKAGE_SRC.resolve() not in imported.parents: raise RuntimeError(f'import isolation violation: {imported}')
print('Source lock PASS:', head)


In [ ]:
if RUN_TESTS:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PACKAGE_SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    r = subprocess.run([sys.executable,'-m','pytest',str(PACKAGE/'tests'),'-q'], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout)
    if r.returncode != 0: raise RuntimeError(f'pytest failed: {r.returncode}')
    print('Full package pytest PASS before data run')


In [ ]:
import numpy as np, scipy, sklearn
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': np.__version__,
    'scipy': scipy.__version__,
    'sklearn': sklearn.__version__,
    'source_sha': SOURCE_SHA,
}
print('Environment:', json.dumps(environment, indent=2))
print('Working disk:', shutil.disk_usage('/kaggle/working'))


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

if not TRAIN_CSV.is_file(): raise FileNotFoundError(TRAIN_CSV)
candidates = sorted(Path('/kaggle/input').rglob('e01_dictionary.npz'))
matches = [p for p in candidates if sha256(p) == EXPECTED_DICTIONARY_SHA256]
print('dictionary candidates:', [str(p) for p in candidates])
if len(matches) != 1:
    raise RuntimeError(f'need exactly one attached v533 e01_dictionary.npz with SHA {EXPECTED_DICTIONARY_SHA256}; matches={matches}')
DICTIONARY_NPZ = matches[0]
print('v533 dictionary lock PASS:', DICTIONARY_NPZ)
print('Train-only input configured:', TRAIN_CSV)


In [ ]:
import contextlib, threading, time
@contextlib.contextmanager
def heartbeat(label, interval_seconds=180):
    stop = threading.Event(); started = time.time()
    def worker():
        while not stop.wait(interval_seconds):
            print(f'[heartbeat] {label}: {(time.time()-started)/60:.1f} min', flush=True)
    t = threading.Thread(target=worker, daemon=True); t.start()
    try: yield
    finally:
        stop.set(); t.join(timeout=1)
        print(f'[heartbeat] {label}: complete {(time.time()-started)/60:.1f} min', flush=True)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ENTRY = PROJECT / 'notebooks' / 'crs-train-entry.py'
if RUN_TRAIN_STAGE:
    command = [sys.executable, str(ENTRY), '--train-csv', str(TRAIN_CSV), '--dictionary-npz', str(DICTIONARY_NPZ), '--output-dir', str(OUTPUT_DIR)]
    with heartbeat('CRS Train-only stage'):
        subprocess.run(command, check=True)


In [ ]:
summary_path = OUTPUT_DIR / 'crs_train_summary.json'
model_path = OUTPUT_DIR / 'crs_train_model.npz'
if not summary_path.is_file() or not model_path.is_file(): raise RuntimeError('missing registered Train artifacts')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
if summary.get('status') != 'TRAIN_STAGE_COMPLETE_PUBLIC_STILL_LOCKED': raise RuntimeError(summary.get('status'))
if summary.get('public_test_accessed') is not False: raise RuntimeError('unexpected validation access flag')
if summary.get('private_test_accessed') is not False: raise RuntimeError('unexpected final-test access flag')
if summary.get('source_dictionary_sha256') != EXPECTED_DICTIONARY_SHA256: raise RuntimeError('dictionary provenance mismatch in summary')
print(json.dumps({
    'status': summary['status'],
    'source_sha': SOURCE_SHA,
    'model_artifact_sha256': summary['model_artifact_sha256'],
    'dictionary_fit': summary['dictionary_fit'],
    'train_diagnostics_only': summary['train_diagnostics_only'],
}, indent=2))
execution_manifest = {
    'issue': 84,
    'draft_pr': 85,
    'source_sha': SOURCE_SHA,
    'train_summary_sha256': sha256(summary_path),
    'train_model_sha256': sha256(model_path),
    'environment': environment,
}
manifest_path = OUTPUT_DIR / 'crs_train_execution_manifest.json'
manifest_path.write_text(json.dumps(execution_manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('Execution manifest:', manifest_path)
print('CRS Train-only stage complete. Validation remains locked.')
